In [64]:
# Modules Import
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, classification_report,
                             roc_curve, roc_auc_score)

In [43]:
# Data Import
cleaned_data = pd.read_csv("../data/cleaned_data.csv")
clean_data = cleaned_data.copy()

In [44]:
# Removing Extra column that added Automatically
clean_data = clean_data.drop(columns = "Unnamed: 0")

In [45]:
# Encoding Male and Female values with 0 and 1
clean_data["sex"] = clean_data["sex"].map({"Male": 1, "Female": 0})

In [46]:
# Encoding True - 1 False - 0
for col in ["fbs", "exang"]:
    clean_data[col] = clean_data[col].astype(int)

encoded_data = pd.get_dummies(clean_data, columns=["cp", "restecg", "slope"],
                            drop_first=True, dtype=int)

In [47]:
# Checking Dataset
print(encoded_data.shape)
encoded_data.head()

(920, 17)


,age,sex,trestbps,chol,fbs,thalch,exang,oldpeak,target,cp_atypical angina,cp_non-anginal,cp_typical angina,restecg_normal,restecg_st-t abnormality,slope_flat,slope_unknown,slope_upsloping
0,63,1,145.0,233.0,1,150.0,0,2.3,0,0,0,1,0,0,0,0,0
1,67,1,160.0,286.0,0,108.0,1,1.5,1,0,0,0,0,0,1,0,0
2,67,1,120.0,229.0,0,129.0,1,2.6,1,0,0,0,0,0,1,0,0
3,37,1,130.0,250.0,0,187.0,0,3.5,0,0,1,0,1,0,0,0,0
4,41,0,130.0,204.0,0,172.0,0,1.4,0,1,0,0,0,0,0,0,1


In [51]:
# Spliting Dataset into Train and Test
X = encoded_data.drop(columns=["target"]) # Symptoms
y = encoded_data["target"] # Corresponding Results

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(2))


(736, 16) (184, 16)
target
1    0.55
0    0.45
Name: proportion, dtype: float64


In [53]:
# Scaling Dat
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train mean (modati 5):", X_train_scaled.mean(axis=0).round(2)[:5])
print("Train std  (modati 5):", X_train_scaled.std(axis=0).round(2)[:5])

Train mean (modati 5): [ 0.  0.  0.  0. -0.]
Train std  (modati 5): [1. 1. 1. 1. 1.]


In [57]:
# Training Actual Machine Learning Model with LogisticRegression
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)
y_prob_log = log_model.predict_proba(X_test_scaled)[:, 1]



In [58]:
# Training Actual Machine Learning Model with DecisionTreeClassifier
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)
y_prob_tree = tree_model.predict_proba(X_test)[:, 1]


In [60]:
# Checking Metrics
metrics_data = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_log),
        "Precision": precision_score(y_test, y_pred_log),
        "Recall": recall_score(y_test, y_pred_log),
        "ROC AUC": roc_auc_score(y_test, y_prob_log),
    },
    {
        "Model": "Decision Tree",
        "Accuracy": accuracy_score(y_test, y_pred_tree),
        "Precision": precision_score(y_test, y_pred_tree),
        "Recall": recall_score(y_test, y_pred_tree),
        "ROC AUC": roc_auc_score(y_test, y_prob_tree),
    }
]).round(4)
metrics_data

,Model,Accuracy,Precision,Recall,ROC AUC
0,Logistic Regression,0.8098,0.8073,0.8627,0.8965
1,Decision Tree,0.7826,0.8163,0.7843,0.8657


In [65]:
# Exporting Models
artifacts = {
    "log_model": log_model,
    "tree_model": tree_model,
    "scaler": scaler,
    "feature_columns": list(X_train.columns),
    "metrics_df": metrics_data,
}

joblib.dump(artifacts, "../models/heart_models.pkl")
print("Exported log_model and tree_model to heart_models.pkl!")

Exported log_model and tree_model to heart_models.pkl!
